# FingerBall Stage 2G — Release-focused Hand Viewer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2G_Release_Focused_Viewer.ipynb)

**188〜193フレーム**を中心に、2D重畳動画・画像原点の疑似3D・手のローカル疑似3D・指関節の2D投影角度を**同じフレームで切り替える**トレーナー向けビューア。

- 投球手の左右は**手動指定**（MediaPipeの左右分類の揺れに依存しない）。3Dで左右を完全検証したという意味ではありません。
- 手ローカル姿勢は手首、中指MCP、示指MCP、小指MCPの**疑似3D**座標から直交基底を計算。**手掌面・手背面の真の3D計測ではありません**。
- 関節角は**画像平面上の3点角度**。PIP/DIPの真の屈伸角ではなく、手の回旋・遠近方向の動きや遮蔽の影響を受けます。
- ボール検出・正確なリリース時刻の推定は行いません。190は動画提供者の目視による推定付近です。
- 動画の外部共有には撮影許諾と個人情報確認が必要です。
**表示方針：** 3D骨格上には指名テキストを重ねず、色分けと凡例で示します。


## 0. セットアップ

In [ ]:
%pip -q install "plotly>=5.18" pandas numpy matplotlib opencv-python-headless ipywidgets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from IPython.display import display,HTML
import cv2,io,base64,json,zipfile,math
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

## 1. 入力ファイルと表示範囲

デフォルトではStage 2Dの21点CSVと、Stage 2Dの`release_zoom_175_210.mp4`を使用します。後者はすでに2D骨格を描画した拡大動画です。元動画で表示する場合は`VIDEO_PATH`と`VIDEO_FRAME_OFFSET=0`を指定してください。

In [ ]:
PROJECT_PATH='/content/drive/MyDrive/母指回内トラッキング'
SOURCE_STAGE='stage2d'             # 'stage2d' or 'stage2e'
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
CSV_PATH=''                        # blank: auto-find Stage2D/2E hand_21_landmarks_long.csv
VIDEO_PATH=''                      # blank: auto-find Stage2D release_zoom_175_210.mp4
VIDEO_FRAME_OFFSET=175             # zoom_175_210.mp4 starts at original frame 175; original MP4 -> 0
THROWING_HAND='Left'               # manually confirmed 'Left' or 'Right'
FIRST_FRAME,LAST_FRAME=188,193
RELEASE_REFERENCE_FRAME=190        # visual estimate only, not auto detected
Z_SCALE=1.0                       # relative Z rendering factor only
GENERATE_PREVIEW_MP4=True
PREVIEW_FPS=3
assert THROWING_HAND in ('Left','Right')
project=Path(PROJECT_PATH)
data_folder=project/f'FingerBall_v2_{SOURCE_STAGE}_results'/VIDEO_STEM
if CSV_PATH:
    csv_path=Path(CSV_PATH)
else:
    csv_path=data_folder/'hand_21_landmarks_long.csv'
    if not csv_path.is_file():
        matches=sorted((project/f'FingerBall_v2_{SOURCE_STAGE}_results').glob('*/hand_21_landmarks_long.csv'))
        if not matches:raise FileNotFoundError('21点CSVが見つかりません。CSV_PATHを指定してください')
        csv_path=matches[0]
if VIDEO_PATH:
    video_path=Path(VIDEO_PATH)
else:
    video_path=project/'FingerBall_v2_stage2d_results'/csv_path.parent.name/'release_zoom_175_210.mp4'
if not csv_path.is_file():raise FileNotFoundError(csv_path)
out_dir=project/'FingerBall_v2_stage2g_results'/csv_path.parent.name
out_dir.mkdir(parents=True,exist_ok=True)
print('CSV',csv_path)
print('Video',video_path, 'found:',video_path.is_file())
print('Manual throwing hand:',THROWING_HAND)

## 2. CSV読込・データQC・2D投影角の計算

In [ ]:
df=pd.read_csv(csv_path)
needed={'frame','landmark_id','x_px','y_px','z_relative','detected'}
if needed-set(df.columns):raise ValueError('Missing columns: '+str(needed-set(df.columns)))
if df.duplicated(['frame','landmark_id']).any():raise ValueError('Duplicate frame/landmark_id')
for k in ('x_px','y_px','z_relative','detected'):
    df[k]=pd.to_numeric(df[k],errors='coerce')
W,H=1206.,648.
for axis,normalized in (('x_px','x_norm'),('y_px','y_norm')):
    if normalized in df:
        s=df[[axis,normalized]].dropna()
        s=s[s[normalized].abs()>1e-6]
        if len(s):
            size=float(np.median(s[axis]/s[normalized]))
            if axis=='x_px':W=size
            else:H=size
if not (np.isfinite(W) and W>0 and np.isfinite(H) and H>0):
    raise ValueError('Invalid image dimensions')
FRAME_IDS=list(range(FIRST_FRAME,LAST_FRAME+1))
xy_by_frame={};pseudo_by_frame={};qc=[]
for f in FRAME_IDS:
    sub=df[(df.frame==f)&df.landmark_id.between(0,20)].sort_values('landmark_id')
    valid=(len(sub)==21 and sub.landmark_id.tolist()==list(range(21)) and
           sub[['x_px','y_px','z_relative','detected']].notna().all().all() and
           (sub.detected.astype(int)==1).all())
    if not valid:
        xy_by_frame[f]=None;pseudo_by_frame[f]=None
    else:
        raw=sub[['x_px','y_px','z_relative']].to_numpy(float)
        xy_by_frame[f]=raw[:,:2]
        pseudo_by_frame[f]=np.column_stack([raw[:,0]/W,raw[:,1]/W,
                             -(raw[:,2]-raw[0,2])*Z_SCALE])
    qc.append({'frame':f,'detected_21':int(valid),
               'hand_label_in_csv':str(sub.handedness_reported.iloc[0])
                  if valid and 'handedness_reported' in sub.columns else '',
               'manually_declared_throwing_hand':THROWING_HAND})
qc_df=pd.DataFrame(qc)
def planar_angle(points,a,b,c):
    v=points[a]-points[b];u=points[c]-points[b]
    nv=np.linalg.norm(v);nu=np.linalg.norm(u)
    if min(nv,nu)<1e-8:return np.nan
    return float(np.degrees(np.arccos(np.clip(np.dot(v,u)/(nv*nu),-1,1))))
ANGLE_SPECS={
 'Index PIP (2D)':(5,6,7),
 'Index DIP (2D)':(6,7,8),
 'Middle PIP (2D)':(9,10,11),
 'Middle DIP (2D)':(10,11,12),
 'Thumb IP (2D)':(2,3,4)}
angle_rows=[]
for f in FRAME_IDS:
    row={'frame':f}
    for name,triplet in ANGLE_SPECS.items():
        row[name]=np.nan if xy_by_frame[f] is None else planar_angle(xy_by_frame[f],*triplet)
    angle_rows.append(row)
angle_df=pd.DataFrame(angle_rows)
angle_csv=out_dir/'release_2D_projected_joint_angles.csv'
qc_csv=out_dir/'release_detection_qc.csv'
angle_df.to_csv(angle_csv,index=False)
qc_df.to_csv(qc_csv,index=False)
print(qc_df.to_string(index=False))
print(angle_df.round(1).to_string(index=False))
print('These angles are 2D projections, not true finger flexion/extension angles.')

## 3. 手ローカル疑似3D座標系

手首0、中指MCP9、示指MCP5、小指MCP17を使い手の向きをそろえます。手の解剖学的左右は手動指定のラベルであり、軸の向きだけで左手と証明するものではありません。退化したフレームは欠損にします。

In [ ]:
local_by_frame={}
def local_palm_frame(p):
    origin=p[0].copy()
    # distal direction towards middle MCP
    distal=p[9]-origin
    dn=np.linalg.norm(distal)
    if dn<1e-7:return None
    ex=distal/dn
    across=p[17]-p[5]          # index-to-little MCP
    across=across-np.dot(across,ex)*ex
    an=np.linalg.norm(across)
    if an<1e-7:return None
    ey=across/an
    ez=np.cross(ex,ey)
    zn=np.linalg.norm(ez)
    if zn<1e-7:return None
    ez=ez/zn
    basis=np.stack([ex,ey,ez],axis=1)
    return (p-origin)@basis
for f in FRAME_IDS:
    p=pseudo_by_frame[f]
    local_by_frame[f]=None if p is None else local_palm_frame(p)
print('Hand-local valid frames:',sum(x is not None for x in local_by_frame.values()),'/',len(FRAME_IDS))

## 4. 2D動画フレームの取得

入力がStage 2Dの`release_zoom_175_210.mp4`ならすでに骨格線が描画されています。元の未注釈MP4を指定したときだけ、元画像座標に21点を重ねます。動画が無い場合は2D静止画欄を説明画像に置き換えます。

In [ ]:
FINGER_SETS={
 'Thumb':[0,1,2,3,4], 'Index':[0,5,6,7,8],
 'Middle':[0,9,10,11,12], 'Ring':[0,13,14,15,16],
 'Little':[0,17,18,19,20]}
FINGER_COLORS={'Thumb':'#f59e0b','Index':'#06b6d4','Middle':'#d946ef',
               'Ring':'#bfc6d1','Little':'#a3a3a3'}
FINGER_BGR={'Thumb':(0,165,255),'Index':(212,182,6),'Middle':(239,70,217),
            'Ring':(208,198,191),'Little':(163,163,163)}
images={};image_data_uri={}
cap=cv2.VideoCapture(str(video_path)) if video_path.is_file() else None
def fetch_video_frame(i):
    if cap is None or not cap.isOpened():return None
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(i-VIDEO_FRAME_OFFSET))
    ok,frame=cap.read()
    return frame if ok else None
for f in FRAME_IDS:
    frame=fetch_video_frame(f)
    if frame is None:
        frame=np.zeros((500,800,3),dtype=np.uint8)
        cv2.putText(frame,f'Frame {f}: video unavailable',(30,250),
                    cv2.FONT_HERSHEY_SIMPLEX,.8,(255,255,255),2)
    elif VIDEO_FRAME_OFFSET==0 and xy_by_frame[f] is not None:
        p=xy_by_frame[f]
        for finger,ids in FINGER_SETS.items():
            color=FINGER_BGR[finger]
            for a,b in zip(ids[:-1],ids[1:]):
                cv2.line(frame,tuple(np.round(p[a]).astype(int)),
                         tuple(np.round(p[b]).astype(int)),color,2)
        for point in p:
            cv2.circle(frame,tuple(np.round(point).astype(int)),2,(255,255,255),-1)
    cv2.putText(frame,f'Original frame {f}',(12,35),cv2.FONT_HERSHEY_SIMPLEX,
                .85,(255,255,255),2)
    images[f]=frame
    # scale embedded image to keep standalone HTML compact
    h,w=frame.shape[:2]
    ratio=min(1.,800/max(h,w))
    preview=cv2.resize(frame,(round(w*ratio),round(h*ratio))) if ratio<1 else frame
    ok,b=cv2.imencode('.jpg',preview,[cv2.IMWRITE_JPEG_QUALITY,85])
    if not ok:raise RuntimeError('JPEG encoding failed')
    image_data_uri[f]='data:image/jpeg;base64,'+base64.b64encode(b).decode('ascii')
if cap is not None:cap.release()
print('2D reference images:',len(image_data_uri))

## 5. 同期した画像原点3Dと手ローカル3DをPlotlyで表示

手ローカル側は手を見やすくするために自動で拡大し、位置軸は全フレーム固定です。関節マーカーは全21点同サイズ。示指と中指は**線だけ色分け**します。

In [ ]:
def traces(f):
    output=[]
    for mode in ('image','local'):
        p=pseudo_by_frame[f] if mode=='image' else local_by_frame[f]
        scene='scene' if mode=='image' else 'scene2'
        for finger,ids in FINGER_SETS.items():
            xx=[];yy=[];zz=[]
            for a,b in zip(ids[:-1],ids[1:]):
                if p is None:
                    xx.extend([None,None,None]);yy.extend([None,None,None]);zz.extend([None,None,None])
                else:
                    xx.extend([float(p[a,0]),float(p[b,0]),None])
                    yy.extend([float(p[a,1]),float(p[b,1]),None])
                    zz.extend([float(p[a,2]),float(p[b,2]),None])
            output.append(go.Scatter3d(x=xx,y=yy,z=zz,mode='lines',scene=scene,
                line=dict(color=FINGER_COLORS[finger],width=7),
                name=finger,showlegend=mode=='local'))
        output.append(go.Scatter3d(
            x=[] if p is None else p[:,0],
            y=[] if p is None else p[:,1],
            z=[] if p is None else p[:,2],
            scene=scene,mode='markers',marker=dict(size=3,color='#d1d5db'),
            name='21 joints',showlegend=False))
    return output
all_local=[v for v in local_by_frame.values() if v is not None]
L=max(.04,max(float(np.max(np.abs(p))) for p in all_local)*1.25) if all_local else .15
all_z=[v[:,2] for v in pseudo_by_frame.values() if v is not None]
Z=max(.04,max(float(np.max(np.abs(z))) for z in all_z)*1.5)
first=FRAME_IDS[0]
fig=go.Figure(data=traces(first),frames=[
    go.Frame(name=str(f),data=traces(f)) for f in FRAME_IDS])
fig.update_layout(
    template='plotly_dark',height=580,
    title=f'Release-focused pseudo-3D | Throwing hand: {THROWING_HAND} (manual)',
    margin=dict(l=10,r=10,t=65,b=10),
    scene=dict(domain=dict(x=[0,.49],y=[0,1]),
       xaxis=dict(title='Image X',range=[-.03,1.03]),
       yaxis=dict(title='Image Y',range=[-.03,1.03]),
       zaxis=dict(title='Wrist-relative Z',range=[-Z,Z]),
       aspectmode='manual',aspectratio=dict(x=1,y=H/W,z=.6),
       camera=dict(eye=dict(x=.1,y=.1,z=2))),
    scene2=dict(domain=dict(x=[.51,1],y=[0,1]),
       xaxis=dict(title='Distal',range=[-L,L]),
       yaxis=dict(title='Index-to-little',range=[-L,L]),
       zaxis=dict(title='Estimated palm normal',range=[-L,L]),
       aspectmode='cube',camera=dict(eye=dict(x=0,y=0,z=2))),
    sliders=[dict(currentvalue=dict(prefix='Frame '),steps=[
        dict(label=str(f),method='animate',args=[[str(f)],
             dict(mode='immediate',frame=dict(duration=0,redraw=True),
                  transition=dict(duration=0))]) for f in FRAME_IDS])],
    updatemenus=[dict(type='buttons',x=.46,y=1.15,direction='left',
      buttons=[
       dict(label='Play',method='animate',args=[None,
          dict(frame=dict(duration=500,redraw=True),fromcurrent=True)]),
       dict(label='Pause',method='animate',args=[[None],
          dict(mode='immediate',frame=dict(duration=0,redraw=True))])])])
fig.show(config=dict(displaylogo=False,scrollZoom=True))

## 6. 関節角度グラフと同期HTML

時間スライダーで2D参照動画フレーム、3D2面、2D投影関節角度グラフを同時切替。

In [ ]:
angle_fig=go.Figure()
linecolors={'Index PIP (2D)':'#06b6d4','Index DIP (2D)':'#67e8f9',
            'Middle PIP (2D)':'#d946ef','Middle DIP (2D)':'#f0abfc',
            'Thumb IP (2D)':'#f59e0b'}
for col,color in linecolors.items():
    angle_fig.add_trace(go.Scatter(x=angle_df.frame,y=angle_df[col],
       mode='lines+markers',line=dict(color=color,width=2),
       marker=dict(size=6),name=col,connectgaps=False))
angle_fig.update_layout(
    template='plotly_dark',height=360,
    title='2D projected angle (near 180 degrees means straight in image plane)',
    xaxis=dict(title='Frame',range=[FIRST_FRAME-.3,LAST_FRAME+.3],dtick=1),
    yaxis=dict(title='2D included angle [deg]',range=[0,185]),
    shapes=[dict(type='line',x0=first,x1=first,y0=0,y1=185,
       line=dict(color='white',width=2,dash='dash'))])
viewer_html=pio.to_html(fig,include_plotlyjs=True,full_html=False,
    div_id='fbRelease3D',config=dict(displaylogo=False,scrollZoom=True))
angle_html=pio.to_html(angle_fig,include_plotlyjs=False,full_html=False,
    div_id='fbAngles')
metric_map={}
for row in angle_rows:
    metric_map[str(row['frame'])]={name:(None if not np.isfinite(value) else round(float(value),1))
       for name,value in row.items() if name!='frame'}
qc_map={str(int(row['frame'])):{'valid':int(row['detected_21']),
    'handedness':row['hand_label_in_csv']} for row in qc}
images_map={str(f):image_data_uri[f] for f in FRAME_IDS}
html_path=out_dir/'FingerBall_Stage2G_release_focused_viewer.html'
header='''<!DOCTYPE html><html lang="ja"><head><meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>FingerBall Stage 2G Release Viewer</title>
<style>body{background:#0f172a;color:#e5e7eb;font:14px sans-serif;padding:18px}
.panel{background:#111827;border:1px solid #334155;border-radius:10px;
padding:10px;margin:10px 0}img{display:block;max-width:100%;max-height:570px;margin:auto}
h2{font-size:18px}#fbInfo{font-size:12px;line-height:1.7}
</style></head><body>
<h2>Release-focused Hand Viewer</h2><p>Throwing hand: __HAND__ (manual).
Visual release reference: frame __REFERENCE__ (not automatically detected).
XY=image coordinates; Z=wrist-relative single-view estimate.
Joint angles are 2D image projections, not clinical flexion angles.</p>
<div class="panel"><h2>2D reference image</h2><img id="fbStill" alt="2D reference"/><div id="fbInfo"></div></div>
<div class="panel"><h2>Camera-image / palm-local pseudo-3D skeleton</h2>'''
footer='''</div><div class="panel"><h2>2D projected finger angles</h2>__ANGLES__</div>
<p>Research preview only. Not calibrated metric 3D. No automated ball-release detection.</p>
__SCRIPT__</body></html>'''
js_script=r'''<script>(function(){
const pics=__PICS__,metrics=__METRICS__,qc=__QC__;
const plot=document.getElementById('fbRelease3D');
const angles=document.getElementById('fbAngles');
const img=document.getElementById('fbStill');
const info=document.getElementById('fbInfo');
function show(f){
  f=String(f);if(pics[f])img.src=pics[f];
  const m=metrics[f]||{},q=qc[f]||{};
  info.textContent='Frame '+f+' | landmarks: '+(q.valid?'present':'missing')+
    ' | classifier handedness: '+(q.handedness||'unknown')+' | '+
    Object.entries(m).map(([k,v])=>k+': '+(v===null?'N/A':v+'°')).join(' / ');
  Plotly.relayout(angles,{'shapes[0].x0':Number(f),'shapes[0].x1':Number(f)});
}
plot.on('plotly_sliderchange',e=>{if(e.step)show(e.step.label)});
plot.on('plotly_animated',()=>{const s=plot._fullLayout.sliders[0];
  if(s&&s.steps[s.active])show(s.steps[s.active].label)});
show('__FIRST__');
})();</script>'''
js_script=js_script.replace('__PICS__',json.dumps(images_map))
js_script=js_script.replace('__METRICS__',json.dumps(metric_map))
js_script=js_script.replace('__QC__',json.dumps(qc_map))
js_script=js_script.replace('__FIRST__',str(first))
html=header.replace('__HAND__',THROWING_HAND).replace('__REFERENCE__',str(RELEASE_REFERENCE_FRAME))
html+=viewer_html+footer.replace('__ANGLES__',angle_html).replace('__SCRIPT__',js_script)
html_path.write_text(html,encoding='utf-8')
print('HTML:',html_path,'size:',round(html_path.stat().st_size/1e6,2),'MB')

## 7. 2D動画と手ローカル疑似3Dの比較MP4・代表PNG

In [ ]:
def render_comparison(f):
    figure=plt.figure(figsize=(12,5),dpi=100,facecolor='#111827')
    a=figure.add_subplot(121)
    b=figure.add_subplot(122,projection='3d')
    a.imshow(cv2.cvtColor(images[f],cv2.COLOR_BGR2RGB))
    a.set_title('2D reference / frame '+str(f),color='white')
    a.axis('off')
    p=local_by_frame[f]
    if p is not None:
        for finger,ids in FINGER_SETS.items():
            b.plot(p[ids,0],p[ids,1],p[ids,2],
                   color=FINGER_COLORS[finger],linewidth=2.6)
        b.scatter(p[:,0],p[:,1],p[:,2],s=14,color='white')
    b.set(xlim=(-L,L),ylim=(-L,L),zlim=(-L,L))
    b.set_box_aspect((1,1,1))
    b.view_init(elev=23,azim=45)
    b.set_title('Palm-local pseudo-3D',color='white')
    b.tick_params(colors='white',labelsize=7)
    figure.suptitle('Throwing hand: '+THROWING_HAND+' (manual) | Projected 2D angles only',
                   color='white',fontsize=12)
    figure.tight_layout()
    figure.canvas.draw()
    rgb=np.asarray(figure.canvas.buffer_rgba())[:,:,:3].copy()
    plt.close(figure)
    return cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)
representative=RELEASE_REFERENCE_FRAME
if representative not in FRAME_IDS:representative=FRAME_IDS[len(FRAME_IDS)//2]
panel_png=out_dir/f'frame_{representative}_2D_and_palm_local.png'
cv2.imwrite(str(panel_png),render_comparison(representative))
preview_mp4=out_dir/'FingerBall_Stage2G_release_comparison.mp4'
if GENERATE_PREVIEW_MP4:
    sample=render_comparison(FRAME_IDS[0]); hh,ww=sample.shape[:2]
    writer=cv2.VideoWriter(str(preview_mp4),cv2.VideoWriter_fourcc(*'mp4v'),
                          PREVIEW_FPS,(ww,hh))
    if not writer.isOpened():raise RuntimeError('Failed to open MP4 writer')
    try:
        for f in FRAME_IDS:
            writer.write(render_comparison(f))
    finally:
        writer.release()
print('PNG:',panel_png)
if GENERATE_PREVIEW_MP4: print('MP4:',preview_mp4)

## 8. 提供者・トレーナー向けZIPと解析条件

In [ ]:
report={
  'stage':'2G','csv':str(csv_path),
  'video':str(video_path) if video_path.is_file() else None,
  'frame_range':[FIRST_FRAME,LAST_FRAME],
  'release_reference_frame_visual_only':RELEASE_REFERENCE_FRAME,
  'manually_specified_throwing_hand':THROWING_HAND,
  'valid_frames':int(qc_df.detected_21.sum()),
  'coordinate_definition':'XY=image coordinates, Z=wrist-relative MediaPipe estimate',
  'local_axes':'wrist-to-middle MCP; index-to-little MCP; their cross product',
  'joint_angle_definition':'2D projected three-point included angle (degrees)',
  'limitations':[
    'Not calibrated 3D or MediaPipe world landmarks',
    'Handedness manually specified, not verified from reconstructed shape',
    'Angles are projected 2D angles, not clinical flexion/extension angles',
    'Out-of-plane movement and ball occlusion may distort digit landmarks',
    'No automatic ball release detection']}
report_path=out_dir/'stage2g_report.json'
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
readme=out_dir/'README_for_video_provider_stage2g.txt'
readme.write_text(
    'FingerBall Stage 2G — 投球リリース付近の手指骨格可視化\n'
    +'対象フレーム: '+str(FIRST_FRAME)+'〜'+str(LAST_FRAME)+'\n'
    +'投球手: '+THROWING_HAND+'（手動指定）\n'
    +'リリースの参考フレーム: '+str(RELEASE_REFERENCE_FRAME)+'（目視推定）\n'
    +'HTMLで時間スライダーを操作すると2D画像・画像座標疑似3D・手ローカル疑似3D・'
    +'示指/中指/母指の2D投影角度が同期します。\n'
    +'3Dは相対Z由来で実空間計測ではありません。関節角度は2D投影角です。\n'
    +'研究開発段階であり、指先やボールの接触時刻は判定していません。\n'
    +'外部共有前に動画利用許諾と個人情報を確認してください。\n',
    encoding='utf-8')
package=out_dir/'FingerBall_Stage2G_release_focused_package.zip'
items=[html_path,angle_csv,qc_csv,panel_png,report_path,readme]
if GENERATE_PREVIEW_MP4:items.append(preview_mp4)
with zipfile.ZipFile(package,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for item in items:
        if not item.is_file():raise FileNotFoundError(item)
        z.write(item,arcname=item.name)
print('ZIP:',package,round(package.stat().st_size/1e6,2),'MB')
print('Results:',out_dir)

## 実行後に見るポイント

1. **190フレーム前後で2Dと3Dの示指・中指が解剖学的に対応しているか**確認してください。
2. 図内の「Left/Right」は手動で与えた情報です。MediaPipeの左右ラベルが揺れても自動で正しい手に直るわけではありません。
3. **関節角度は2D投影角**です。見かけの伸展変化を実際のPIP/DIP屈伸角に直結させないでください。
4. フレームごとの再構成が不自然なら、Stage 2DのROI・指の割当て・相対Zを先に修正してください。より信頼性の高い奥行きには`hand_world_landmarks`の再出力と検証が必要です。